In [1]:
!pip install gdal
# !pip install vdal

You should consider upgrading via the '/opt/conda/bin/python3.7 -m pip install --upgrade pip' command.


In [2]:
# !pip install shapely
!pip install alphashape

     |████████████████████████████████| 1.0 MB 13.6 MB/s 
     |████████████████████████████████| 694 kB 53.4 MB/s 
     |████████████████████████████████| 1.9 MB 42.9 MB/s 
  Attempting uninstall: rtree
    Found existing installation: Rtree 0.9.4
    Uninstalling Rtree-0.9.4:
      Successfully uninstalled Rtree-0.9.4
  Attempting uninstall: networkx
    Found existing installation: networkx 2.4
    Uninstalling networkx-2.4:
      Successfully uninstalled networkx-2.4
ERROR: After October 2020 you may experience errors when installing or updating packages. This is because pip will change the way that it resolves dependency conflicts.

We recommend you use --use-feature=2020-resolver to test your packages with the new resolver before it becomes the default.

trimesh 4.4.1 requires numpy>=1.20, but you'll have numpy 1.18.5 which is incompatible.
You should consider upgrading via the '/opt/conda/bin/python3.7 -m pip install --upgrade pip' command.


In [3]:
!pip install geopandas

You should consider upgrading via the '/opt/conda/bin/python3.7 -m pip install --upgrade pip' command.


In [4]:
import os
import gdal
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import pickle
import shapely
import csv

# File paths

In [5]:
dataset = 'Kamlapur'
# dataset = 'mirpur_technical'
# dataset = 'narinda'
datasrc = '/kaggle/input/mirpur-technical/'
orthophoto_path = None
point_cloud_path = None
dsm_path = None
label_base_path  = None

# import laspy

# import pdal

# Define input paths
if dataset == 'Kamlapur':
    orthophoto_path = datasrc + 'Kamlapur Dataset/Kamlapur Dataset/Kamlapur/Orthophoto/Komlapur_Orthophoto.tif'
#     point_cloud_path = datasrc + 'Kamlapur Dataset/Kamlapur Dataset/Kamlapur/Point Cloud/combined_point_cloud.las'
    dsm_path = datasrc + 'Kamlapur Dataset/Kamlapur Dataset/Kamlapur/DSM/Komlapur_dsm.tif'
    label_base_path = datasrc + 'Kamlapur Dataset/Kamlapur Dataset/Komlapur_Shape/'
    
elif dataset == 'mirpur_technical':
    orthophoto_path = datasrc + 'Mirpur Technical Dataset/Mirpur Technical Dataset/Mirpur_Technical/Orthophoto/Mirpur_Technical_Orthophoto.tif'
#     point_cloud_path = datasrc + 'Mirpur Technical Dataset/Mirpur Technical Dataset/Mirpur_Technical/Point Cloud/combined_point_cloud.las'
    dsm_path = datasrc + 'Mirpur Technical Dataset/Mirpur Technical Dataset/Mirpur_Technical/DSM/Mirpur_Technical_dsm.tif'
    label_base_path = datasrc + 'Mirpur Technical Dataset/Mirpur Technical Dataset/Mirpur_Shape/'
    
elif dataset == 'narinda':
    orthophoto_path = datasrc + 'Narinda Dataset/Narinda Dataset/Narianda/Orthophoto/Narinda_Orthophoto.tif'
#     point_cloud_path = datasrc + 'Mirpur Technical Dataset/Mirpur Technical Dataset/Mirpur_Technical/Point Cloud/combined_point_cloud.las'
    dsm_path = datasrc + 'Narinda Dataset/Narinda Dataset/Narinda/DSM/Narinda_1_dsm.tif'
    label_base_path = datasrc + 'Narinda Dataset/Narinda Dataset/Narinda_shapes/Narinda/'

# Define tile size (in pixels)
tile_size = 512



# Declarations

In [6]:
def get_np_array(path):
    demo = gpd.read_file(path)
    coords = demo.geometry.apply (lambda p: (p.x, p.y))
    array = np.array (list (coords))
    return array

In [7]:
# Dot labelled informations
ac_path = label_base_path + 'AC.shp'
ac_arr = get_np_array(ac_path)
# dengue_habitat_path = label_base_path + 'Dengue _habitat.shp'
flower_pot_path = label_base_path + 'Flower_tob.shp'
flower_pot_arr = get_np_array(flower_pot_path)

glass_ware_path = label_base_path + 'Glass_ware.shp'
glass_ware_arr = get_np_array(glass_ware_path)

green_coconut_path = label_base_path + 'Green_coconut.shp'
green_coconut_arr = get_np_array(green_coconut_path)

open_tank_path = label_base_path + 'Open_tank.shp'
open_tank_arr = get_np_array(open_tank_path)

others_path = label_base_path + 'Others.shp'
others_arr = get_np_array(others_path)

polythene_path = label_base_path + 'Polythene.shp'
polythene_arr = get_np_array(polythene_path)

reservoir_path = label_base_path + 'Reservior.shp'
reservoir_arr = get_np_array(reservoir_path)

tyres_path = label_base_path + 'Tyres.shp'
tyres_arr = get_np_array(tyres_path)

# print(tyres_arr)

# demo = gpd.read_file(tyres_path)
# for index, row in demo.iterrows():
#         longitude = row.geometry.x
#         latitude = row.geometry.y
#         print(longitude, latitude)
    
# Demo == Represents all the annotated points (for tyres)
# demo.shape[0] == number of entries

ortho_demo = gdal.Open(orthophoto_path)
geotransform = ortho_demo.GetGeoTransform()
# Calculate the origin (top left) coordinates
origin_x = geotransform[0]
origin_y = geotransform[3]# print(origin_x, origin_y)

pixel_width = geotransform[1]
pixel_height = geotransform[5]
# print(pixel_width, pixel_height)

# print(ortho_demo.RasterXSize, ortho_demo.RasterYSize)

# About GDAL and geotransform variables

Geospatial Data Abstraction Library  

The geotransform is a **tuple of six values** that define the relationship between pixel coordinates (raster space) and world coordinates (geographic or projected space) for a georeferenced raster dataset. It is an important concept in GIS and remote sensing applications, as it allows you to transform between pixel coordinates and real-world coordinates. The values in the geotransform have specific meanings:

**Top Left X (geotransform[0]):**

This is the x-coordinate of the top left corner of the raster dataset. It represents the horizontal distance from the origin of the spatial reference system to the left edge of the dataset.

**Pixel Width (geotransform[1]):**

This value represents the width of a single pixel in the x-direction. It is typically expressed in the units of the spatial reference (e.g., meters, degrees, etc.). The width is positive if the dataset is oriented from left to right.

**Rotation (geotransform[2]):**

This value represents any rotation of the raster. It's usually zero in most cases.

**Top Left Y (geotransform[3]):**

This is the y-coordinate of the top left corner of the raster dataset. It represents the vertical distance from the origin of the spatial reference system to the top edge of the dataset. It's negative because the top-left corner is often considered the origin.

**Rotation (geotransform[4]):**

This value represents any rotation of the raster. It's usually zero in most cases.

**Pixel Height (geotransform[5]):**

This value represents the height of a single pixel in the y-direction. It is typically negative (if the dataset is oriented from top to bottom) and expressed in the units of the spatial reference. The absolute value of the pixel height is used to compute distances.


***In summary:***

***(geotransform[0], geotransform[3]) represents the coordinates of the top left corner of the dataset.
(geotransform[1], geotransform[5]) represents the size of a single pixel in the x and y directions.
(geotransform[2], geotransform[4]) represents any rotation, which is typically zero.***

The geotransform essentially defines a linear transformation from pixel space to the spatial reference space, allowing you to convert pixel coordinates to geographic or projected coordinates.

# Making Tiles

**gdal.Translate:** This is a function from the GDAL library that is used to translate (copy or extract) a portion of a raster dataset into a new file.

**tile_path:** This is the path to the output file where the extracted tile will be saved.

**dataset:** This is the original raster dataset from which the tile will be extracted.

**srcWin=[x, y, tile_size, tile_size]:** This parameter defines the source window from which the tile will be extracted. The srcWin parameter takes four values:

**x:** The starting X coordinate (column) of the source window in the original dataset.

**y:** The starting Y coordinate (row) of the source window in the original dataset.

**tile_size:** The width and height of the source window (and consequently, the size of the extracted tile).

When you call gdal.Translate with the provided parameters, it reads the specified source window from the original raster dataset (dataset) and saves it as a new tile in the location specified by tile_path. The source window is defined by the starting X and Y coordinates (x and y) and has dimensions of tile_size by tile_size.

This line essentially performs the operation of extracting a tile from a larger raster dataset and saving it as a separate file. The extracted tile will be tile_size by tile_size pixels and will correspond to the region defined by the source window coordinates x and y.

In [8]:
# generate_tiles(orthophoto_path, tile_size)

# # Step 3: Alignment (manual or automatic as needed)

# # Step 4: Verification (visual inspection)

# # Step 5: Further Processing

# # Step 6: Metadata Preservation (store tile metadata)


# Generate tiles from orthophoto

**Transposed the binary image to match the shape with the orthophoto. Need to inspect it more though if it is right**

Yep, it's right

**Longitude == x and latitude == y**

In [9]:
!rm -rf /kaggle/working/*
# Step 1: Preprocessing (already georeferenced)
base_path = '/kaggle/working/tiles/'

if not os.path.exists(base_path):
    os.mkdir(base_path)
#     os.mkdir(base_path + dataset +'/')
# Step 2: Tile Generation

In [10]:
objmap = {}
objnamemap = {}

objlist = ['ac', 'flower_pot', 'glass_ware', 'green_coconut', 'open_tank', 'others', 'polythene', 'reservoir', 'tyres']

for obj in objlist:
    objmap[obj] = 0
    objnamemap[obj] = 0

In [11]:
def in_range(arr, geo_info):
#     geo_info[0] = min_long, geo_info[1] = min_lat, geo_info[2] = max_long, geo_info[3] = max_lat
    count = 0
    for d in arr:
        if geo_info[0] <= d[0] <= geo_info[2] and geo_info[1] <= d[1] <= geo_info[3]:
            count += 1
    return count

# flower_pot, open_tank, polythene, reservoir, tyres
classes = np.array([1, 4, 6, 7, 8]) # Classes to consider
classes_arr = np.array([ac_arr, flower_pot_arr, glass_ware_arr, green_coconut_arr, open_tank_arr, others_arr, polythene_arr, reservoir_arr, tyres_arr])

def save_tile(geo_info):
    #  If no class in a tile, return false
    result = False
    for idx in classes:
        obj_count = in_range(classes_arr[idx], geo_info)
        if obj_count > 0:
            objmap[objlist[idx]] += obj_count
            objnamemap[objlist[idx]] = 1
            result = True;
        obj_count = 0
    return result

In [12]:
def pixel2geocoor( x, y ):
    tile_min_longitude = origin_x + x * pixel_width
    tile_max_latitude = origin_y + y * pixel_height
    tile_max_longitude = tile_min_longitude + tile_size * pixel_width
    tile_min_latitude = tile_max_latitude + tile_size * pixel_height
    
    return tile_min_longitude, tile_min_latitude, tile_max_longitude, tile_max_latitude


In [13]:
# def create_csv(shp_file):
#     csv_file_path = os.path.join(label_path, shp_file + '.csv')

#     # Create an empty CSV file with headers
#     with open(csv_file_path, mode='w', newline='') as file:
#         writer = csv.writer(file)
#         writer.writerow(['image_id', 'latitude', 'longitude'])  # Replace with your column names
#     file.close()
        
# def append_to_csv(csv_file_path, data):
#     with open( os.path.join( label_path ,csv_file_path ), mode='a', newline='') as file:
# #         print(data)
#         writer = csv.writer(file )
#         writer.writerow([ data['image_id'], data['latitude'], data['longitude'] ])
#     file.close()

In [14]:
import os
import numpy as np
from osgeo import gdal

def generate_tiles_from_ortho(ortho_image, orthophoto_path, tile_size, base_path):
    dataset = gdal.Open(orthophoto_path)
    
    width_ortho = dataset.RasterXSize
    height_ortho = dataset.RasterYSize
    geotransform = dataset.GetGeoTransform()
    origin_x, origin_y = geotransform[0], geotransform[3]
    pixel_width, pixel_height = geotransform[1], geotransform[5]

    print("Ortho image dimension:", width_ortho, height_ortho)
    print("Origin x:", origin_x, "Origin y:", origin_y)
    print("Pixel width:", pixel_width, "Pixel height:", pixel_height)

    height, width = ortho_image.shape[:2]
    count, black_count = 0, 0

    for y in range(0, height, int(tile_size * 0.25)):
        tile_height = tile_size if y + tile_size <= height else height - y
        if tile_height <= 0:
            continue

        for x in range(0, width, int(tile_size * 0.25)):
            tile_width = tile_size if x + tile_size <= width else width - x
            if tile_width <= 0:
                continue

            block = ortho_image[y:y + tile_height, x:x + tile_width]

            geo_info = pixel2geocoor(x, y)

            black_pixels = np.sum(block == 0)
            total_pixels = block.size
            if black_pixels / total_pixels > 0.80 and not save_tile(geo_info):
                black_count += 1
                print(f"Skipping tile at ({x}, {y}) - more than 80% black.")
                continue

            if not save_tile(geo_info):
                continue

            count += 1
            filename = ''

            for i in [1, 4, 6, 7, 8]:
                if objnamemap[objlist[i]] == 1:
                    filename += f'_{objlist[i]}'
                    objnamemap[objlist[i]] = 0

            tile_path = f'tile_{geo_info[0]}_{geo_info[1]}_{x}_{y}{filename}.tif'
            print(f"{count}: Creating tile at ({x}, {y})")
            gdal.Translate(os.path.join(base_path, tile_path), dataset, srcWin=[x, y, tile_width, tile_height])

    print('Total tiles generated:', count)
    print('Total black tiles skipped:', black_count)
    for obj in objlist:
        print(obj, ':', objmap[obj])


In [15]:
ortho_image = ortho_demo.GetRasterBand(1).ReadAsArray()
print("Generating tile size:", 2048)
tile_size = 2048
generate_tiles_from_ortho(ortho_image, orthophoto_path, tile_size, base_path)

Generating tile size: 2048
Ortho image dimension: 16460 14590
Origin x: 237842.57878 Origin y: 2627504.1153700002
Pixel width: 0.026240000000000003 Pixel height: -0.026240000000000003
Skipping tile at (0, 0) - more than 80% black.
Skipping tile at (512, 0) - more than 80% black.
Skipping tile at (1024, 0) - more than 80% black.
Skipping tile at (1536, 0) - more than 80% black.
Skipping tile at (2048, 0) - more than 80% black.
Skipping tile at (2560, 0) - more than 80% black.
Skipping tile at (3072, 0) - more than 80% black.
Skipping tile at (3584, 0) - more than 80% black.
1: Creating tile at (4096, 0)
2: Creating tile at (4608, 0)
3: Creating tile at (5120, 0)
4: Creating tile at (5632, 0)
5: Creating tile at (6144, 0)
6: Creating tile at (6656, 0)
7: Creating tile at (7680, 0)
8: Creating tile at (8192, 0)
9: Creating tile at (8704, 0)
10: Creating tile at (9216, 0)
11: Creating tile at (9728, 0)
12: Creating tile at (10240, 0)
13: Creating tile at (10752, 0)
Skipping tile at (13312,

In [16]:
# ortho_array = ortho_demo.GetRasterBand(1).ReadAsArray()
# print(ortho_demo.RasterXSize, ortho_demo.RasterYSize)
# print(ortho_array.shape)
# zerocnt = 0
# nonzerocnt = 0
# for i in range(len(ortho_array)):
#     for j in range(len(ortho_array[i])):
#         if ortho_array[i,j] == 0: zerocnt += 1
#         else: nonzerocnt += 1
# print(zerocnt, nonzerocnt)


In [17]:
# Clear directory contents
# !rm -rf /kaggle/working/*

# **Converting tif to jpg**

In [18]:
# from PIL import Image
# from pathlib import Path

In [19]:
from PIL import Image

tif_dir = base_path
jpg_dir = "/kaggle/working/tiles_JPG"

os.makedirs(jpg_dir, exist_ok=True)

# Loop through all files in the TIF directory
for filename in os.listdir(tif_dir):
    if filename.endswith(".tif") or filename.endswith(".TIF"):
        tif_path = os.path.join(tif_dir, filename)
        image = Image.open(tif_path)

        rgb_image = image.convert("RGB")

        jpg_filename = os.path.splitext(filename)[0] + ".jpg"
        jpg_path = os.path.join(jpg_dir, jpg_filename)

        # Save the image as JPEG
        rgb_image.save(jpg_path, "JPEG")

print("Conversion completed successfully.")

Conversion completed successfully.


In [20]:
# !rm -rf /kaggle/working/tiles